In [ ]:
%pip install -q langchain langchain-community langchain-core langchain-openai langgraph --upgrade
%pip install -q python-dotenv nest_asyncio

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
from dotenv import load_dotenv
from langchain_community.document_loaders.pdf import ZeroxPDFLoader

load_dotenv()

loader = ZeroxPDFLoader(
    file_path="./real_estate_tax.pdf",
    model="gpt-4.1-mini",
)

pages = loader.load()

In [ ]:
print(f"total pages: {len(pages)}")
print(pages[0].page_content[:500])

In [ ]:
with open("real_estate_tax.txt", "w") as f:
    for page in pages:
        f.write(page.page_content + "\n\n")

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph


class AgentState(TypedDict):
    query: str  # user question
    answer: str  # ai answer (세율)
    tax_base_equation: str  # 과세표준 계산 수식
    tax_deduction: str  # 공제액
    market_ratio: str  # 공정시장가액비율
    tax_base: str  # 과세표준 계산


graph_builder = StateGraph(AgentState)

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embedding_function = OpenAIEmbeddings(model="text-embedding-3-large")

vector_store = Chroma(
    embedding_function=embedding_function,
    collection_name="real_estate_tax",
    persist_directory="./real_estate_tax_collection",
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})